In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
%%bash
mkdir -p out_rev/pyqg

In [3]:
import dabench as dab
import numpy as np
import jax
from timeit import default_timer as timer
import pandas as pd
import jaxlib
import matplotlib.pyplot as plt

from ray import train, tune
from hyperopt import hp
from ray.tune.search.hyperopt import HyperOptSearch
import pickle

# Define parameters

In [4]:
year_in_timesteps = 4380
spinup_size = 5*year_in_timesteps
valid_size = round(year_in_timesteps/4)
transient_size = 1*year_in_timesteps
test_size = 1*year_in_timesteps

In [5]:
nr_steps = spinup_size + valid_size + transient_size + test_size
delta_t=7200
analysis_window = 6*delta_t

### Function definition: ETKF

In [6]:
def run_etkf(system_dim_xy, nr_steps, spinup_size, valid_size, test_size,
                        test_run, delta_t, sigma_bg_multiplier, sigma_obs_multiplier, 
                        analysis_window,
                        random_seed, ensemble_size):
    
    np_rng = np.random.default_rng(random_seed)
    jax.clear_backends()

    ### Nature Run
    nature_run = dab.data.PyQGJax(nx=system_dim_xy, ny=system_dim_xy, delta_t=delta_t, 
                                  store_as_jax=True, random_seed=random_seed)

    nature_run.generate(n_steps=nr_steps) 
    nr_spinup, nr_valid, nr_test = nature_run.split_train_valid_test(
        spinup_size-(test_size), test_size, 0)

    if not test_run:
        nr_eval = nr_valid
    else:
        nr_eval = nr_test
        
        
    ### Observations
    obs_location_count = round(nature_run.system_dim/2)

    # First we need to calculate the per-variable SD for QGS model
    obs_sd_scale = 0.1
    per_variable_sd = np.std(nr_spinup.values, axis=0)
    obs_sd = 0.1*per_variable_sd

    obs_pyqg = dab.observer.Observer(
        nr_eval,
        time_indices = np.arange(0, nr_eval.time_dim, 3),
        random_location_count = obs_location_count,
        error_bias = 0.0,
        error_sd = obs_sd,
        random_seed=random_seed+test_run,
        stationary_observers=True,
        store_as_jax=True
    )

    obs_vec_pyqg = obs_pyqg.observe()

    
    ### Forecast Model
    model_pyqg = dab.data.PyQGJax(nx=system_dim_xy, ny=system_dim_xy,
                                  store_as_jax=True, random_seed=random_seed)

    class PyQGModel(dab.model.Model):                                                                       
        """Defines model wrapper for forecasting."""
        def forecast(self, state_vec, n_steps):
            gridded_values = state_vec.values.reshape(self.model_obj.original_dim)
            self.model_obj.generate(x0=gridded_values, n_steps=n_steps)
            new_vals = self.model_obj.values

            new_vec = dab.vector.StateVector(values=new_vals, store_as_jax=True)

            return new_vec

        def _forecast_x0(self, x0, n_steps):
            self.model_obj.generate(x0=x0.reshape(self.model_obj.original_dim),
                                    n_steps=n_steps)
            return self.model_obj.values
        
    fc_model = PyQGModel(model_obj=model_pyqg)

    
    ### Set up DA matrices: H (observation), R (obs error), B (background error)
    obs_times_per_window=2
    total_obs_count = obs_times_per_window*obs_location_count
    sigma_obs=sigma_obs_multiplier*obs_sd[obs_vec_pyqg.location_indices[0]]
    sigma_bg = sigma_bg_multiplier*obs_sd
    H = np.zeros((total_obs_count, nature_run.system_dim))
    H[np.arange(H.shape[0]), np.tile(obs_vec_pyqg.location_indices[0],obs_times_per_window)] = 1
    R = (np.tile(sigma_obs, obs_times_per_window)**2)* np.identity(total_obs_count)
    B = (sigma_bg**2)*np.identity(nature_run.system_dim)
    
    da_time_start = timer()

    # Prep DA
    dc = dab.dacycler.ETKF(
        system_dim=nature_run.system_dim,
        delta_t=nr_eval.delta_t,
        B=B,
        R=R,
        H=H,
        model_obj=fc_model,
        ensemble_dim=ensemble_size,
        multiplicative_inflation=1.01
        )

    # Generate initial conditions
    cur_tstep = 0
    x0_original = nr_eval.values[cur_tstep] + np_rng.normal(size=(ensemble_size, nature_run.system_dim,), 
                                                            scale=sigma_bg)

    x0_sv = dab.vector.StateVector(
        values=x0_original,
        store_as_jax=True)
    
    # Execute
    out_statevec = dc.cycle(
        input_state = x0_sv,
        start_time = nr_eval.times[cur_tstep],
        obs_vector = obs_vec_pyqg,
        analysis_window=analysis_window,
        n_cycles=int(nr_eval.time_dim/6) - 2,
        return_forecast=True,
        obs_error_sd=sigma_obs
    )

    da_time = timer()-da_time_start

    return out_statevec, obs_vec_pyqg, nr_eval, da_time

# Run

In [ ]:
ensemble_size=250
out_dict_list_etkf = []
system_dim_xy_list = [16, 24, 32]
test_run=False

for system_dim_xy in system_dim_xy_list:
    
    random_seed = system_dim_xy
    
    run_dict = dict(
        system_dim_xy=system_dim_xy, 
        nr_steps=nr_steps,
        spinup_size=spinup_size,
        valid_size=valid_size,
        test_size=test_size,
        test_run=test_run,
        delta_t=delta_t,
        sigma_bg_multiplier=0.5,
        sigma_obs_multiplier=1.25,
        analysis_window=analysis_window,
        random_seed=random_seed,
        ensemble_size=ensemble_size
    )
    
    out_etkf, obs_vec_pyqg, nr_eval, da_time = run_etkf(**run_dict)
    run_dict['time'] = da_time
    run_dict['run_num'] = 0
    out_file = './out_rev/pyqg/pyqg_enkf_statevec_{}dim_v1.pkl'.format(system_dim_xy)
    with open(out_file, 'wb') as f: 
        pickle.dump(out_etkf, f) 
    f.close()

        
    print('Run {}, Time = {}'.format(0,run_dict['time']))
    out_dict_list_etkf.append(run_dict)

/tmp/ipykernel_296763/2246864022.py:7: DeprecationWarning: jax.clear_backends is deprecated.
  jax.clear_backends()


Initial condition not set. Start with random IC.
Run 0, Time = 1032.5766168310074


/tmp/ipykernel_296763/2246864022.py:7: DeprecationWarning: jax.clear_backends is deprecated.
  jax.clear_backends()


Initial condition not set. Start with random IC.
Run 0, Time = 1185.2567943850008


/tmp/ipykernel_296763/2246864022.py:7: DeprecationWarning: jax.clear_backends is deprecated.
  jax.clear_backends()


Initial condition not set. Start with random IC.
Run 0, Time = 1352.8482890680025


In [1]:
fig, axes = plt.subplots(6, 1, sharex = True, figsize = (10, 8))
for i, ax in enumerate(axes):
    j=obs_vec_pyqg.location_indices[0,i]
    ax.plot(out_etkf.times, nr_eval.values[:out_etkf.times.shape[0],j], lw = 3, label = 'True')
    ax.errorbar(out_etkf.times, np.mean(out_etkf.values, axis=1)[:,j],
                yerr=np.ptp(out_etkf.values, axis=1)[:,j], elinewidth=0.5, ecolor='red')
    ax.plot(obs_vec_pyqg.times, obs_vec_pyqg.values[:,i])
    ax.set_ylabel(r'$x_{:d}$'.format(j), fontsize = 16)
ax.set_xlabel('Time (s)')
plt.show()

NameError: name 'plt' is not defined